# 01 - Baseline: LSTM language model (one-hot vs Word2Vec)

**Goal.** Train an LSTM that predicts the next word of English Wikipedia text, and compare two input representations: **one-hot** vs **Word2Vec**.


**Stages (this notebook is built step by step):**
1. Setup and reproducibility  <- *this stage*
2. Data pipeline  <- *this stage*
3. Sanity checks and reference scores  <- *this stage*
4. Model, training with early stopping  *(next)*
5. One-hot vs Word2Vec experiments and results table  *(later)*

## 1. Setup and reproducibility

There are **two different kinds of randomness**, and we treat them differently:

| Seed | Controls | Rule |
|---|---|---|
| `data_seed` (in `CFG`, = 42) | which articles are sampled and how they are split | **Never changes.** Baseline and attention must see identical data. |
| `SEED` (below) | weight initialisation and batch order | Varies between repetitions, so we can report mean and spread. |

`DEBUG = True` uses a tiny subset so you can test code quickly. Set it back to `False` for real results.

In [1]:
import numpy as np
import common
from common import CFG, set_seed, get_device, prepare_data, sanity_checks, reference_perplexities

DEBUG = False                        # True -> tiny subset for fast debugging
cfg = dict(CFG, n_articles=50) if DEBUG else dict(CFG)

SEED = 0                             # model/training seed (data seed is fixed in CFG)
set_seed(SEED)


device = get_device()                # 'cuda' if available, else CPU fallback

print("device:", device)
for k, v in cfg.items():
    print(f"{k:>18}: {v}")

device: cuda
         data_seed: 42
        n_articles: 1000
 min_article_words: 200
        train_frac: 0.8
          val_frac: 0.1
          min_freq: 5
            window: 20
        cache_path: data/articles.json.gz


## 2. Data pipeline

**Corpus.** English Wikipedia (`wikimedia/wikipedia`, snapshot `20231101.en`). According to its Hugging Face dataset card it is licensed CC-BY-SA 3.0 / GFDL (please double-check the card yourself before citing it in the report).

**What `prepare_data` does, in order:**

1. **Sample a subset.** The dataset is streamed, so we do not download all ~6.4M articles. We keep `n_articles` articles with at least `min_article_words` words (to skip stubs). The subset is saved to `data/articles.json.gz`, so every later run, and your teammate, uses the same text. *Note: streaming shuffle is pseudo-random (a buffer of 10,000), not a perfectly uniform sample.*
2. **Tokenize** into words with NLTK. Case is kept, because the GoogleNews Word2Vec vectors are case-sensitive.
3. **Split by article** into 80% / 10% / 10% (train / validation / test). Splitting *by article* (not by window) matters: if neighbouring windows of the same article ended up in different splits, the model would be tested on text it has practically seen. That is **data leakage**.
4. **Build the vocabulary from the training articles only.** Words seen fewer than `min_freq = 5` times become `<UNK>`. This has two purposes: it keeps the vocabulary (and the one-hot size) manageable, and it makes sure `<UNK>` actually appears during training, so the model learns what to do with it.
5. **Make windows.** For a window of 20 words, the input is words *i..i+19* and the target is word *i+20*, i.e. the input **shifted by one**. Windows never cross article boundaries.

The first run downloads and tokenizes the articles, which can take a few minutes. Later runs read the saved file.

In [2]:
data = prepare_data(cfg)
print("X_train:", data["X_train"].shape, " y_train:", data["y_train"].shape)

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

X_train: (750732, 20)  y_train: (750732,)


## 3. Sanity checks and reference scores

Before training anything, we verify the pipeline instead of trusting it. `sanity_checks` asserts that:

- no article appears in two splits (no leakage),
- the vocabulary uses training data only and respects `min_freq`,
- shapes and id ranges are right,
- the number of windows equals what we expect from the article lengths,
- **the target really is the input shifted by one word**,
- `<UNK>` occurs in the training data.

If any check fails, the cell stops with an error. Do not continue past a failure.

**Reference scores.** A perplexity number means little on its own, so we compute two trivial "models":

- **Uniform:** every word has probability 1/V, so the perplexity equals the vocabulary size V.
- **Unigram:** word probabilities from training frequencies, ignoring the context entirely.

An LSTM that cannot clearly beat the unigram reference has learned nothing useful from the context.

In [3]:
stats = sanity_checks(data)

refs = reference_perplexities(data)
print()
print(f"Reference perplexities  (V = {data['vocab_size']})")
print(f"  uniform        : {refs['uniform']:.1f}")
print(f"  unigram (val)  : {refs['unigram_val']:.1f}")
print(f"  unigram (test) : {refs['unigram_test']:.1f}")

vocab_size = 13625   window = 20   min_freq = 5
split  articles    tokens   windows  UNK in text %  UNK as target %
train       800    766732    750732           9.53             9.56
val         100    102816    100816          12.83            12.79
test        100    122557    120557          15.37            15.38
All sanity checks passed.

Reference perplexities  (V = 13625)
  uniform        : 13625.0
  unigram (val)  : 554.9
  unigram (test) : 484.5


In [4]:
# Look at one training example: input words -> target word
i = min(1000, len(data["X_train"]) - 1)
x, y = data["X_train"][i], data["y_train"][i]
print(" ".join(data["id_to_word"][int(t)] for t in x))
print("   --> target:", data["id_to_word"][int(y)])

River West Branch that drains the southern slopes of the <UNK> <UNK> from a <UNK> point at an elevation of
   --> target: ,


### What to check in the output above

- The three splits should have **similar statistics**. The `<UNK>` rate in validation/test is usually a bit higher than in training (the vocabulary was built from training text), but a huge gap would be suspicious.
- The `<UNK>` rate in training should be above 0, which the check enforces.
- Write down `vocab_size` and the unigram reference. They are the yardstick for every model score later.

**Next stage:** the LSTM model, the training loop with early stopping on validation perplexity, and the evaluation function.